In [ ]:
combine the 20-year-data in one master file

rank the junction blacksites in descending order of their years on the blacksite list.

clean the data further: ST = STREET, RD = ROAD

In [ ]:
import pandas as pd
import re

# assume df is already loaded
# df = pd.read_excel("Junction_Blacksite_Master_Q4_2004-2024.xlsx")

def clean_street_suffixes(value):
    if pd.isna(value):
        return value

    s = str(value).strip()
    s = s.upper()

    # normalize common road/street abbreviations
    s = re.sub(r"\bST\.\b", "STREET ", s)
    s = re.sub(r"\bST\b", "STREET", s)
    s = re.sub(r"\bRD\.\b", "ROAD ", s)
    s = re.sub(r"\bRD\b", "ROAD", s)

    # clean double spaces caused by replacements
    s = re.sub(r"\s+", " ", s).strip()

    # optional: remove stray comma spacing
    s = s.replace(" ,", ",")
    s = s.replace(", ", ", ")

    return s

df["street_1"] = df["street_1"].apply(clean_street_suffixes)
df["street_2"] = df["street_2"].apply(clean_street_suffixes)

# quick check
print(df[["street_1", "street_2"]].head())

In [2]:
import pandas as pd
import re

# Read the master workbook
df = pd.read_excel("Junction_Blacksite_Master_Q4_2004-2024.xlsx").copy()

# Clean street names: ST -> STREET, RD -> ROAD
def normalize_street(x):
    if pd.isna(x):
        return ""
    x = str(x).upper().strip()
    x = x.replace(".", "")
    x = x.replace(",", " ")
    x = re.sub(r"\s+", " ", x)
    x = re.sub(r"\bST\b", "STREET", x)
    x = re.sub(r"\bRD\b", "ROAD", x)
    return x.strip()

for col in ["street_1", "street_2"]:
    df[col] = df[col].map(normalize_street)

# Make sure numeric columns are clean
df["site_id"] = pd.to_numeric(df["site_id"], errors="coerce")
df["year"] = pd.to_numeric(df["year"], errors="coerce")

# Sort the cleaned street names so reversed street_1/street_2 pairs share one key.
street_pairs = df.apply(
    lambda row: sorted((row["street_1"], row["street_2"])), axis=1
)
df["_street_1_key"] = street_pairs.map(lambda pair: pair[0])
df["_street_2_key"] = street_pairs.map(lambda pair: pair[1])
df["_region_key"] = df["region"].fillna("").astype(str).str.upper().str.strip()
df["_district_key"] = df["district"].fillna("").astype(str).str.upper().str.strip()
group_cols = ["_region_key", "_district_key", "_street_1_key", "_street_2_key"]

agg = (
    df.groupby(group_cols, as_index=False, sort=False, dropna=False)
      .agg(
          region=("region", "first"),
          district=("district", "first"),
          site_id=("site_id", "first"),
          street_1=("_street_1_key", "first"),
          street_2=("_street_2_key", "first"),
          number_of_years_on_blacksite=("year", "nunique"),
          starting_year_on_blacksite=("year", "min"),
          ending_year_on_blacksite=("year", "max"),
          accidents_total=("accidents_total", "sum"),
          accidents_pedestrians=("accidents_pedestrians", "sum"),
      )
      .drop(columns=group_cols)
      .sort_values(
          ["number_of_years_on_blacksite", "starting_year_on_blacksite"],
          ascending=[False, True],
          kind="mergesort"
      )
      .rename(columns={
          "number_of_years_on_blacksite": "number of years on the blacksite list",
          "starting_year_on_blacksite": "starting year on the blacksite list",
          "ending_year_on_blacksite": "ending year on the blacksite list",
          "accidents_total": "accidents_total(accumulated)",
          "accidents_pedestrians": "accidents_pedestrians(accumulated)",
      })
)

# Keep final output columns in the order requested
final_cols = [
    "region",
    "district",
    "site_id",
    "street_1",
    "street_2",
    "number of years on the blacksite list",
    "starting year on the blacksite list",
    "ending year on the blacksite list",
    "accidents_total(accumulated)",
    "accidents_pedestrians(accumulated)",
]

agg = agg[final_cols]

# Save to Excel
agg.to_excel("Junction_Blacksite_Normalized_Years.xlsx", index=False)

agg.head(20)

,region,district,site_id,street_1,street_2,number of years on the blacksite list,starting year on the blacksite list,ending year on the blacksite list,accidents_total(accumulated),accidents_pedestrians(accumulated)
233,New Territories,Tsuen Wan,50.0,TSUEN WAN ROAD,TSUEN TSING INTERCHANGE,16,2009,2024,311,0
240,Kowloon,Yau Tsim Mong,35.0,CHATHAM ROAD SOUTH,AUSTIN ROAD CHEONG WAN ROAD,16,2009,2024,324,18
250,Kowloon,Yau Tsim Mong,28.0,MONG KOK ROAD,NATHAN ROAD,16,2009,2024,241,26
219,Kowloon,Kowloon City,8.0,WATERLOO ROAD,ARGYLE STREET PRINCESS MARGARET ROAD,15,2009,2024,240,11
251,Kowloon,Yau Tsim Mong,29.0,NATHAN ROAD,ARGYLE STREET,15,2009,2024,236,54
230,Kowloon,Sham Shui Po,14.0,TONKIN STREET,CHEUNG SHA WAN ROAD,12,2009,2023,161,17
232,New Territories,Tsuen Wan,51.0,TSUEN KAM INTERCHANGE,TEXACO ROAD NORTH,12,2009,2023,182,0
261,Kowloon,Yau Tsim Mong,47.0,WATERLOO ROAD,NATHAN ROAD,12,2009,2024,179,25
215,Kowloon,Kowloon City,6.0,MA TAU CHUNG ROAD,SUNG WONG TOI ROAD FU NING STREET,11,2009,2019,165,7
237,Kowloon,Yau Tsim Mong,24.0,ARGYLE STREET,SAI YEE STREET,11,2009,2021,154,34
